In [82]:
!pip install fuzzywuzzy
!pip install geopy
import pandas as pd
import numpy as np
from geopy.distance import geodesic
import re


# 1. Drop duplicate census tracts
# 2. Standardize state names across all datasets
# 3. Convert lat/lon to census tract IDs (use census geocoder or shapefile)
# 4. Handle missing MW/power data (many "Unknown" values)
# 5. Standardize TRUE/FALSE/Yes/No to boolean
# 6. Extract numeric values from strings (e.g., "$14 billion" → 14000000000)
# 7. Validate geographic matches (some may fall outside census boundaries)



In [83]:

# Load datasets
dc_osdc = pd.read_csv('/Users/zdiener/data-center-classification/data/raw/im3_open_source_data_center_atlas.csv')  # Dataset 1
dc_frac = pd.read_csv('/Users/zdiener/data-center-classification/data/raw/Data_Centers_Database - FracTracker Data Centers.csv')  # Dataset 2
dc_frac.head()

,facility_name,address,city,state,zip,county,lat,long,status,location_confidence,...,info_source_1,info_source_2,info_source_3,info_source_4,info_source_5,info_source_6,info_source_7,info_source_8,date_created,date_updated
0,Project Marvel,Rock Mountain Lake Rd,Bessemer,AL,35022.0,Jefferson,33.342000,-87.034100,Proposed,High,...,https://www.datacenterdynamics.com/en/news/700...,https://www.youtube.com/watch?v=ICSrvQ7meow&ab...,https://www.al.com/news/2025/06/14-billion-pro...,https://www.wbrc.com/2025/09/14/bessemer-resid...,https://www.wbrc.com/2025/11/19/bessemer-city-...,https://www.datacenterdynamics.com/en/news/ala...,https://abc3340.com/news/abc-3340-news-iteam/b...,NaN,09/16/2025,03/13/2026
1,DC Blox,433 6th St S,Birmingham,AL,35233.0,Jefferson,33.500510,-86.821000,Operating,High,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08/25/2025,08/25/2025
2,Google Data Center,48809 Alabama 277,Bridgeport,AL,35740.0,Jackson,34.920190,-85.744600,Operating,High,...,https://www.madeinalabama.com/2018/04/google-k...,https://claycorp.com/work/mission-critical,NaN,NaN,NaN,NaN,NaN,NaN,07/15/2025,02/18/2026
3,Western Hospitality Partners Data Center,Childersburg Industrial Park,Childersburg,AL,35044.0,Talladega,33.340885,-86.336205,Proposed,Medium,...,https://www.datacenterdynamics.com/en/news/pla...,https://www.childersburg.org/childersburg-indu...,https://abc3340.com/news/local/concerns-grow-i...,NaN,NaN,NaN,NaN,NaN,01/26/2026,05/03/2026
4,DC Blox,333 Diamond Dr NW,Huntsville,AL,35806.0,Madison,34.710030,-86.693000,Operating,High,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,08/25/2025,08/25/2025


In [84]:
dc_frac.info()

<class 'pandas.DataFrame'>
RangeIndex: 1493 entries, 0 to 1492
Data columns (total 44 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   facility_name              1492 non-null   str    
 1   address                    1435 non-null   str    
 2   city                       1489 non-null   str    
 3   state                      1493 non-null   str    
 4   zip                        1443 non-null   float64
 5   county                     1492 non-null   str    
 6   lat                        1493 non-null   float64
 7   long                       1493 non-null   float64
 8   status                     1493 non-null   str    
 9   location_confidence        1493 non-null   str    
 10  purpose                    64 non-null     str    
 11  operator_name              815 non-null    str    
 12  tenant                     26 non-null     str    
 13  mw                         544 non-null    str    
 14  siz

In [85]:
dc_osdc.info()

<class 'pandas.DataFrame'>
RangeIndex: 1242 entries, 0 to 1241
Data columns (total 13 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   id         1242 non-null   int64  
 1   state      1242 non-null   str    
 2   state_abb  1242 non-null   str    
 3   state_id   1242 non-null   int64  
 4   county     1242 non-null   str    
 5   county_id  1242 non-null   int64  
 6   operator   784 non-null    str    
 7   ref        327 non-null    str    
 8   name       1029 non-null   str    
 9   sqft       1149 non-null   float64
 10  lon        1242 non-null   float64
 11  lat        1242 non-null   float64
 12  type       1242 non-null   str    
dtypes: float64(3), int64(3), str(7)
memory usage: 126.3 KB


In [86]:
print("="*60)
print("OSDC DATASET INITIAL ASSESSMENT")
print("="*60)
print(f"\nShape: {dc_osdc.shape}")
print(f"\nColumns: {list(dc_osdc.columns)}")
print(f"\nData types:\n{dc_osdc.dtypes}")
print(f"\nMissing values:\n{dc_osdc.isnull().sum()}")
print(f"\nDuplicate rows: {dc_osdc.duplicated().sum()}")
print(f"\nSample data:")
print(dc_osdc.head(3))

print("\n" + "="*60)
print("FRACTRACKER DATASET INITIAL ASSESSMENT")
print("="*60)
print(f"\nShape: {dc_frac.shape}")
print(f"\nColumns: {list(dc_frac.columns)}")
print(f"\nData types:\n{dc_frac.dtypes}")
print(f"\nMissing values:\n{dc_frac.isnull().sum()}")
print(f"\nDuplicate rows: {dc_frac.duplicated().sum()}")
print(f"\nSample data:")
print(dc_frac.head(3))


OSDC DATASET INITIAL ASSESSMENT

Shape: (1242, 13)

Columns: ['id', 'state', 'state_abb', 'state_id', 'county', 'county_id', 'operator', 'ref', 'name', 'sqft', 'lon', 'lat', 'type']

Data types:
id             int64
state            str
state_abb        str
state_id       int64
county           str
county_id      int64
operator         str
ref              str
name             str
sqft         float64
lon          float64
lat          float64
type             str
dtype: object

Missing values:
id             0
state          0
state_abb      0
state_id       0
county         0
county_id      0
operator     458
ref          915
name         213
sqft          93
lon            0
lat            0
type           0
dtype: int64

Duplicate rows: 0

Sample data:
        id           state state_abb  state_id            county  county_id  \
0  2744301      New Jersey        NJ        34  Middlesex County         23   
1  7805491            Ohio        OH        39   Franklin County         49 

In [87]:
# A more detailed assessment of each dataset to identify specific cleaning needs.
# 1. Check coordinate validity
print("\n1. COORDINATE VALIDATION:")
print(f"   Latitude range: {dc_osdc['lat'].min():.4f} to {dc_osdc['lat'].max():.4f}")
print(f"   Longitude range: {dc_osdc['lon'].min():.4f} to {dc_osdc['lon'].max():.4f}")
invalid_coords = dc_osdc[
    (dc_osdc['lat'] < 24) | (dc_osdc['lat'] > 50) |
    (dc_osdc['lon'] < -125) | (dc_osdc['lon'] > -65)
]
print(f"   Records with potentially invalid US coordinates: {len(invalid_coords)}")
if len(invalid_coords) > 0:
    print(invalid_coords[['id', 'name', 'state', 'lat', 'lon']].head())

# 2. Check state consistency
print("\n2. STATE DATA:")
print(f"   Unique states: {dc_osdc['state'].nunique()}")
print(f"   Missing state: {dc_osdc['state'].isnull().sum()}")
print(f"   Missing state_abb: {dc_osdc['state_abb'].isnull().sum()}")
print(f"   State distribution (top 10):")
print(dc_osdc['state'].value_counts().head(10))

# 3. Check name/operator data
print("\n3. NAME & OPERATOR DATA:")
print(f"   Missing names: {dc_osdc['name'].isnull().sum()}")
print(f"   Missing operators: {dc_osdc['operator'].isnull().sum()}")
print(f"   Unique operators (non-null): {dc_osdc['operator'].nunique()}")
print(f"   Top operators:")
print(dc_osdc['operator'].value_counts().head(10))

# Look at records with missing names
print(f"\n   Sample records with missing names:")
print(dc_osdc[dc_osdc['name'].isnull()][['id', 'operator', 'state', 'county', 'type']].head())

# 4. Check size data
print("\n4. FACILITY SIZE DATA:")
print(f"   Missing sqft: {dc_osdc['sqft'].isnull().sum()}")
print(f"   Sqft statistics:")
print(dc_osdc['sqft'].describe())
unrealistic_size = dc_osdc[(dc_osdc['sqft'] < 1000) | (dc_osdc['sqft'] > 10000000)]
print(f"   Potentially unrealistic sizes (<1k or >10M sqft): {len(unrealistic_size)}")
if len(unrealistic_size) > 0:
    print(unrealistic_size[['name', 'operator', 'sqft', 'type']].head())

# 5. Check type field
print("\n5. TYPE FIELD:")
print(f"   Unique types: {dc_osdc['type'].unique()}")
print(f"   Type distribution:")
print(dc_osdc['type'].value_counts())

# 6. Check for duplicates by coordinates
print("\n6. DUPLICATE COORDINATE ANALYSIS:")
dup_coords = dc_osdc.groupby(['lat', 'lon']).size()
print(f"   Unique coordinate pairs: {len(dup_coords)}")
print(f"   Facilities with same coordinates: {len(dup_coords[dup_coords > 1])}")
if len(dup_coords[dup_coords > 1]) > 0:
    print("   Examples of shared coordinates:")
    for coords, count in dup_coords[dup_coords > 1].head(3).items():
        print(f"\n      Location {coords}: {count} facilities")
        print(dc_osdc[(dc_osdc['lat'] == coords[0]) & (dc_osdc['lon'] == coords[1])][['id', 'name', 'operator', 'type']])

print("\n" + "="*60)
print("FRACTRACKER DATASET - DETAILED CLEANING ASSESSMENT")
print("="*60)

# 1. Check coordinate validity
print("\n1. COORDINATE VALIDATION:")
print(f"   Latitude range: {dc_frac['lat'].min():.4f} to {dc_frac['lat'].max():.4f}")
print(f"   Longitude range: {dc_frac['long'].min():.4f} to {dc_frac['long'].max():.4f}")
print(f"   Missing coordinates: lat={dc_frac['lat'].isnull().sum()}, long={dc_frac['long'].isnull().sum()}")
invalid_coords_frac = dc_frac[
    (dc_frac['lat'] < 24) | (dc_frac['lat'] > 50) |
    (dc_frac['long'] < -125) | (dc_frac['long'] > -65)
]
print(f"   Records with potentially invalid US coordinates: {len(invalid_coords_frac)}")
if len(invalid_coords_frac) > 0:
    print(invalid_coords_frac[['facility_name', 'state', 'lat', 'long']].head())

# 2. Check state data
print("\n2. STATE DATA:")
print(f"   Unique states: {dc_frac['state'].nunique()}")
print(f"   Missing state: {dc_frac['state'].isnull().sum()}")
print(f"   State distribution:")
print(dc_frac['state'].value_counts())

# 3. Check facility status
print("\n3. FACILITY STATUS:")
print(f"   Missing status: {dc_frac['status'].isnull().sum()}")
print(f"   Status values:")
print(dc_frac['status'].value_counts())

# 4. Check power data (mw is string type!)
print("\n4. POWER DATA (MW):")
print(f"   Missing MW: {dc_frac['mw'].isnull().sum()}")
print(f"   Unique MW values (sample of 20):")
print(dc_frac['mw'].value_counts().head(20))

# 5. Check size data
print("\n5. SIZE DATA:")
print(f"   Missing facility_size_sqft: {dc_frac['facility_size_sqft'].isnull().sum()}")
print(f"   facility_size_sqft is STRING type - sample values:")
print(dc_frac['facility_size_sqft'].value_counts().head(10))

print(f"\n   Sizerank categories:")
print(dc_frac['sizerank'].value_counts())

# 6. Check EJ-relevant fields
print("\n6. ENVIRONMENTAL JUSTICE FIELDS:")
ej_fields = ['community_pushback', 'advocacy_information', 'resistance_status', 
             'community_group_website_1', 'petition_url']
for field in ej_fields:
    non_null = dc_frac[field].notna().sum()
    print(f"   {field}: {non_null} records ({non_null/len(dc_frac)*100:.1f}%)")

print(f"\n   Community pushback values:")
print(dc_frac['community_pushback'].value_counts(dropna=False))

print(f"\n   Resistance status values:")
print(dc_frac['resistance_status'].value_counts(dropna=False))

# 7. Check date fields
print("\n7. DATE FIELDS:")
print(f"   date_created samples:")
print(dc_frac['date_created'].value_counts().head(5))
print(f"\n   date_updated samples:")
print(dc_frac['date_updated'].value_counts().head(5))

# 8. Check location confidence
print("\n8. LOCATION CONFIDENCE:")
print(dc_frac['location_confidence'].value_counts(dropna=False))

# 9. Check for the duplicate row
print("\n9. DUPLICATE ROW:")
dup_mask = dc_frac.duplicated(keep=False)
if dup_mask.sum() > 0:
    print(f"   Found {dup_mask.sum()} duplicate rows:")
    print(dc_frac[dup_mask][['facility_name', 'city', 'state', 'operator_name']])

# 10. Check text fields for cleaning needs
print("\n10. TEXT FIELD CLEANING NEEDS:")
# Check for leading/trailing whitespace
print(f"   Facility names with leading/trailing spaces: {dc_frac['facility_name'].str.strip().ne(dc_frac['facility_name']).sum()}")
print(f"   Operator names with leading/trailing spaces: {dc_frac['operator_name'].str.strip().ne(dc_frac['operator_name']).sum()}")

# Check for missing facility names
print(f"\n   Records with missing facility_name:")
print(dc_frac[dc_frac['facility_name'].isnull()][['address', 'city', 'state', 'operator_name']])

# 11. Check numeric fields that are stored as strings
print("\n11. NUMERIC FIELDS STORED AS STRINGS:")
string_numeric_fields = ['mw', 'facility_size_sqft', 'number_of_generators', 'number_of_buildings']
for field in string_numeric_fields:
    if field in dc_frac.columns:
        print(f"\n   {field} - sample values:")
        print(dc_frac[field].value_counts().head(10))


1. COORDINATE VALIDATION:
   Latitude range: 18.4174 to 48.7573
   Longitude range: -123.3286 to -66.1090
   Records with potentially invalid US coordinates: 1
               id       name        state        lat       lon
1197  10014176940  Microsoft  Puerto Rico  18.417436 -66.10899

2. STATE DATA:
   Unique states: 46
   Missing state: 0
   Missing state_abb: 0
   State distribution (top 10):
state
Virginia      292
California     94
Oregon         94
Texas          92
Washington     81
Iowa           57
Ohio           56
Arizona        54
New Jersey     47
Illinois       41
Name: count, dtype: int64

3. NAME & OPERATOR DATA:
   Missing names: 213
   Missing operators: 458
   Unique operators (non-null): 129
   Top operators:
operator
Amazon Web Services            176
Digital Realty                  70
Meta                            61
Google                          60
Microsoft                       53
Flexential                      42
Equinix                         38
Qualit

   Unique types: <StringArray>
['building', 'campus', 'point']
Length: 3, dtype: str
   Type distribution:
type
building    1040
campus       109
point         93
Name: count, dtype: int64

6. DUPLICATE COORDINATE ANALYSIS:
   Unique coordinate pairs: 1240
   Facilities with same coordinates: 2
   Examples of shared coordinates:

      Location (33.746561602113815, -84.57973805393436): 2 facilities
            id                          name        operator      type
773  975064000  Digital Realty Atlanta ATL11  Digital Realty  building
774  975064000  Digital Realty Atlanta ATL11  Digital Realty  building

      Location (41.4982126945884, -93.78856494092211): 2 facilities
            id            name   operator    type
683  830885779  Project Osmium  Microsoft  campus
684  830885779  Project Osmium  Microsoft  campus

FRACTRACKER DATASET - DETAILED CLEANING ASSESSMENT

1. COORDINATE VALIDATION:
   Latitude range: 21.4805 to 48.2220
   Longitude range: -158.0214 to 92.4045
   Missi

In [88]:
dc_osdc.head()

,id,state,state_abb,state_id,county,county_id,operator,ref,name,sqft,lon,lat,type
0,2744301,New Jersey,NJ,34,Middlesex County,23,NaN,NaN,Verizon,105786.0,-74.496521,40.544256,building
1,7805491,Ohio,OH,39,Franklin County,49,NaN,NaN,Discover Financial Services New Albany,188209.0,-82.814358,40.100657,building
2,9474864,North Carolina,NC,37,Caldwell County,27,Google,NaN,Google Data Center,3407194.0,-81.546515,35.894738,campus
3,13924557,Iowa,IA,19,Polk County,153,Microsoft,NaN,Project Alluvion,10962475.0,-93.711719,41.515955,campus
4,14593270,North Carolina,NC,37,Catawba County,35,NaN,NaN,Apple - Maiden Data Center,5431080.0,-81.261809,35.588771,campus


In [89]:
def clean_fractracker(df):
    """
    Clean FracTracker dataset
    """
    df_clean = df.copy()
    
    print("Starting FracTracker cleaning...")
    print(f"Initial shape: {df_clean.shape}")
    
    # 1. REMOVE DUPLICATE ROWS
    print("\n1. Removing duplicates...")
    initial_count = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    print(f"   Removed {initial_count - len(df_clean)} duplicate rows")
    
    # 2. STANDARDIZE LOCATION CONFIDENCE
    print("\n2. Standardizing location_confidence...")
    confidence_map = {
        'High': 'High',
        'high': 'High',
        'Medium': 'Medium',
        'Medium ': 'Medium',  # trailing space
        'Low': 'Low'
    }
    df_clean['location_confidence'] = df_clean['location_confidence'].map(confidence_map)
    print(f"   Values after standardization:")
    print(f"   {df_clean['location_confidence'].value_counts()}")
    
    # 3. STRIP WHITESPACE FROM TEXT FIELDS
    print("\n3. Stripping whitespace from text fields...")
    text_fields = ['facility_name', 'address', 'city', 'county', 'operator_name', 
                   'status', 'purpose', 'tenant', 'advocacy_information', 
                   'community_pushback', 'resistance_status']
    for field in text_fields:
        if field in df_clean.columns:
            df_clean[field] = df_clean[field].str.strip()
    print(f"   Whitespace stripped from {len(text_fields)} fields")
    
    # 4. CLEAN COMMUNITY_PUSHBACK - Replace "Unknown" with NaN
    print("\n4. Cleaning community_pushback field...")
    print(f"   Values before cleaning:")
    print(f"   {df_clean['community_pushback'].value_counts(dropna=False)}")
    
    # Replace "Unknown" (case-insensitive) with NaN
    df_clean['community_pushback'] = df_clean['community_pushback'].replace({
        'Unknown': np.nan,
        'unknown': np.nan,
        'UNKNOWN': np.nan
    })
    
    print(f"   Values after cleaning:")
    print(f"   {df_clean['community_pushback'].value_counts(dropna=False)}")
    
    # 5. STANDARDIZE SIZERANK
    print("\n5. Standardizing sizerank...")
    # Fix the inconsistency: "Hyperscale (101-999 MW)" -> "Hyperscale (100-999 MW)"
    df_clean['sizerank'] = df_clean['sizerank'].replace({
        'Hyperscale (101-999 MW)': 'Hyperscale (100-999 MW)'
    })
    print(f"   Sizerank values after standardization:")
    print(f"   {df_clean['sizerank'].value_counts()}")
    
    # 6. CLEAN MW FIELD (convert string to numeric)
    print("\n6. Cleaning MW field...")
    def parse_mw(mw_str):
        """Extract numeric MW value from string"""
        if pd.isna(mw_str):
            return np.nan
        
        mw_str = str(mw_str).strip()
        
        # Handle ranges (e.g., "100-200", "150-1,000")
        range_match = re.search(r'(\d+(?:,\d+)?)\s*-\s*(\d+(?:,\d+)?)', mw_str)
        if range_match:
            # Take the midpoint of the range
            low = float(range_match.group(1).replace(',', ''))
            high = float(range_match.group(2).replace(',', ''))
            return (low + high) / 2
        
        # Extract first number found
        number_match = re.search(r'(\d+(?:,\d+)*(?:\.\d+)?)', mw_str)
        if number_match:
            return float(number_match.group(1).replace(',', ''))
        
        return np.nan
    
    df_clean['mw_numeric'] = df_clean['mw'].apply(parse_mw)
    df_clean['mw_original'] = df_clean['mw']  # Keep original for reference
    print(f"   Converted {df_clean['mw_numeric'].notna().sum()} MW values to numeric")
    if df_clean['mw_numeric'].notna().sum() > 0:
        print(f"   MW range: {df_clean['mw_numeric'].min():.1f} to {df_clean['mw_numeric'].max():.1f}")
    
    # 7. CLEAN FACILITY_SIZE_SQFT (convert string to numeric)
    print("\n7. Cleaning facility_size_sqft...")
    def parse_sqft(sqft_str):
        """Extract numeric sqft value from string"""
        if pd.isna(sqft_str):
            return np.nan
        
        sqft_str = str(sqft_str).strip()
        
        # Remove commas and extract number
        number_match = re.search(r'(\d+(?:,\d+)*(?:\.\d+)?)', sqft_str)
        if number_match:
            return float(number_match.group(1).replace(',', ''))
        
        return np.nan
    
    df_clean['facility_size_sqft_numeric'] = df_clean['facility_size_sqft'].apply(parse_sqft)
    df_clean['facility_size_sqft_original'] = df_clean['facility_size_sqft']  # Keep original
    print(f"   Converted {df_clean['facility_size_sqft_numeric'].notna().sum()} sqft values to numeric")
    if df_clean['facility_size_sqft_numeric'].notna().sum() > 0:
        print(f"   Sqft range: {df_clean['facility_size_sqft_numeric'].min():.0f} to {df_clean['facility_size_sqft_numeric'].max():.0f}")
    
    # 8. CLEAN NUMBER_OF_GENERATORS
    print("\n8. Cleaning number_of_generators...")
    def parse_generators(gen_str):
        """Extract numeric generator count from string"""
        if pd.isna(gen_str):
            return np.nan
        
        gen_str = str(gen_str).strip()
        
        # Handle ranges (e.g., "12-24")
        range_match = re.search(r'^(\d+)\s*-\s*(\d+)$', gen_str)
        if range_match:
            low = int(range_match.group(1))
            high = int(range_match.group(2))
            return (low + high) / 2
        
        # Extract first number
        number_match = re.search(r'(\d+)', gen_str)
        if number_match:
            return int(number_match.group(1))
        
        return np.nan
    
    df_clean['number_of_generators_numeric'] = df_clean['number_of_generators'].apply(parse_generators)
    df_clean['number_of_generators_original'] = df_clean['number_of_generators']  # Keep original
    print(f"   Converted {df_clean['number_of_generators_numeric'].notna().sum()} generator values to numeric")
    
    # 9. CLEAN NUMBER_OF_BUILDINGS
    print("\n9. Cleaning number_of_buildings...")
    def parse_buildings(bldg_str):
        """Extract numeric building count from string"""
        if pd.isna(bldg_str):
            return np.nan
        
        bldg_str = str(bldg_str).strip()
        
        # Extract first number
        number_match = re.search(r'(\d+)', bldg_str)
        if number_match:
            return int(number_match.group(1))
        
        return np.nan
    
    df_clean['number_of_buildings_numeric'] = df_clean['number_of_buildings'].apply(parse_buildings)
    df_clean['number_of_buildings_original'] = df_clean['number_of_buildings']  # Keep original
    print(f"   Converted {df_clean['number_of_buildings_numeric'].notna().sum()} building values to numeric")
    
    # 10. PARSE DATE FIELDS
    print("\n10. Parsing date fields...")
    df_clean['date_created_parsed'] = pd.to_datetime(df_clean['date_created'], format='%m/%d/%Y', errors='coerce')
    df_clean['date_updated_parsed'] = pd.to_datetime(df_clean['date_updated'], format='%m/%d/%Y', errors='coerce')
    print(f"   Parsed {df_clean['date_created_parsed'].notna().sum()} created dates")
    print(f"   Parsed {df_clean['date_updated_parsed'].notna().sum()} updated dates")
    
    # 11. FIX MISSING FACILITY NAME
    print("\n11. Handling missing facility names...")
    # For the row with coordinates as facility_name
    mask_bad_name = df_clean['facility_name'].str.match(r'^\d+\.\d+$', na=False)
    if mask_bad_name.any():
        print(f"   Found {mask_bad_name.sum()} records with coordinates as facility_name")
        # Use operator name or create placeholder
        df_clean.loc[mask_bad_name, 'facility_name'] = df_clean.loc[mask_bad_name, 'operator_name'].fillna('Unknown Facility')
    
    # Handle truly missing facility names
    missing_names = df_clean['facility_name'].isna()
    if missing_names.any():
        print(f"   Found {missing_names.sum()} records with missing facility_name")
        df_clean.loc[missing_names, 'facility_name'] = df_clean.loc[missing_names, 'operator_name'].fillna('Unknown Facility')
    
    # 12. VALIDATE COORDINATES AND CHECK STATE/COORDINATE CONSISTENCY
    print("\n12. Validating coordinates...")
    
    # Define expanded valid ranges for all US states and territories
    # Contiguous US: roughly 24-50°N, -125 to -65°W
    # Alaska: 51-72°N, -180 to -130°W
    # Hawaii: 19-23°N, -161 to -154°W
    
    # Check for completely invalid coordinates (outside all US territories)
    invalid_coords = (
        (df_clean['lat'] < 18) | (df_clean['lat'] > 72) |  # Outside US latitude range
        (df_clean['long'] < -180) | (df_clean['long'] > -60)  # Outside US longitude range
    )
    
    if invalid_coords.any():
        print(f"   ERROR: {invalid_coords.sum()} records with invalid coordinates (outside US)")
        print(df_clean[invalid_coords][['facility_name', 'state', 'lat', 'long']])
    
    # Check for state/coordinate mismatches
    print("\n   Checking state/coordinate consistency...")
    
    # Define approximate state coordinate ranges
    state_bounds = {
        'HI': {'lat_min': 18, 'lat_max': 23, 'lon_min': -161, 'lon_max': -154},
        'AK': {'lat_min': 51, 'lat_max': 72, 'lon_min': -180, 'lon_max': -130},
        # Contiguous states
        'default': {'lat_min': 24, 'lat_max': 50, 'lon_min': -125, 'lon_max': -65}
    }
    
    def check_state_coord_mismatch(row):
        """Check if coordinates match the state"""
        state = row['state']
        lat = row['lat']
        lon = row['long']
        
        if state == 'HI':
            bounds = state_bounds['HI']
        elif state == 'AK':
            bounds = state_bounds['AK']
        else:
            bounds = state_bounds['default']
        
        in_range = (bounds['lat_min'] <= lat <= bounds['lat_max'] and 
                   bounds['lon_min'] <= lon <= bounds['lon_max'])
        
        return not in_range
    
    mismatches = df_clean.apply(check_state_coord_mismatch, axis=1)
    
    if mismatches.any():
        print(f"   WARNING: {mismatches.sum()} records with state/coordinate mismatches")
        print(df_clean[mismatches][['facility_name', 'state', 'county', 'lat', 'long']])
        print("\n   These may need manual review - coordinates might be correct but state code wrong, or vice versa")
    
    # 13. CREATE HELPER FLAGS
    print("\n13. Creating data quality flags...")
    df_clean['has_mw_data'] = df_clean['mw_numeric'].notna()
    df_clean['has_size_data'] = df_clean['facility_size_sqft_numeric'].notna()
    df_clean['has_ej_data'] = (
        df_clean['community_pushback'].notna() |
        df_clean['advocacy_information'].notna() |
        df_clean['resistance_status'].notna()
    )
    df_clean['has_community_pushback'] = df_clean['community_pushback'].notna()
    df_clean['is_proposed'] = df_clean['status'].str.contains('Proposed', case=False, na=False)
    df_clean['is_operating'] = df_clean['status'].str.contains('Operating', case=False, na=False)
    
    print(f"\nFinal shape: {df_clean.shape}")
    print(f"Records with MW data: {df_clean['has_mw_data'].sum()}")
    print(f"Records with size data: {df_clean['has_size_data'].sum()}")
    print(f"Records with EJ data: {df_clean['has_ej_data'].sum()}")
    print(f"Records with community pushback (Yes/No): {df_clean['has_community_pushback'].sum()}")
    
    return df_clean

In [92]:
# Run the cleaning
dc_frac_clean = clean_fractracker(dc_frac)

dc_frac_clean = dc_frac_clean.rename(columns={
    'long': 'longitude',
    'lat': 'latitude',
    'state': 'state_abb'  
})

# Save cleaned version
dc_frac_clean.to_csv('/Users/zdiener/data-center-classification/data/processed/fractracker_cleaned.csv', index=False)
print("\nCleaned FracTracker data saved to: data/processed/fractracker_cleaned.csv")

# Show summary of cleaned data
print("\n" + "="*60)
print("CLEANING SUMMARY")
print("="*60)
print(f"\nOriginal records: {len(dc_frac)}")
print(f"Cleaned records: {len(dc_frac_clean)}")
print(f"\nCommunity pushback breakdown:")
print(dc_frac_clean['community_pushback'].value_counts(dropna=False))


Starting FracTracker cleaning...
Initial shape: (1493, 44)

1. Removing duplicates...
   Removed 1 duplicate rows

2. Standardizing location_confidence...
   Values after standardization:
   location_confidence
High      1188
Medium     253
Low         48
Name: count, dtype: int64

3. Stripping whitespace from text fields...
   Whitespace stripped from 11 fields

4. Cleaning community_pushback field...
   Values before cleaning:
   community_pushback
Unknown    1268
Yes         212
NaN          12
Name: count, dtype: int64
   Values after cleaning:
   community_pushback
NaN    1280
Yes     212
Name: count, dtype: int64

5. Standardizing sizerank...
   Sizerank values after standardization:
   sizerank
Unknown                    850
Hyperscale (100-999 MW)    312
Medium (11-50 MW)          105
Mega campus (>1,000 MW)     98
Small (0-10 MW)             95
Large (51-99 MW)            32
Name: count, dtype: int64

6. Cleaning MW field...
   Converted 544 MW values to numeric
   MW range: 0

### cleaning the ocsd dataset 

In [93]:
def clean_osdc(df):
    """
    Clean OSDC (Open Source Data Center) dataset
    """
    df_clean = df.copy()
    
    print("Starting OSDC cleaning...")
    print(f"Initial shape: {df_clean.shape}")
    
    # 1. REMOVE DUPLICATE ROWS (if any)
    print("\n1. Checking for duplicates...")
    initial_count = len(df_clean)
    df_clean = df_clean.drop_duplicates()
    print(f"   Removed {initial_count - len(df_clean)} duplicate rows")
    
    # 2. STRIP WHITESPACE FROM TEXT FIELDS
    print("\n2. Stripping whitespace from text fields...")
    text_fields = ['state', 'state_abb', 'county', 'operator', 'ref', 'name', 'type']
    for field in text_fields:
        if field in df_clean.columns and df_clean[field].dtype == 'object':
            df_clean[field] = df_clean[field].str.strip()
    print(f"   Whitespace stripped from {len(text_fields)} fields")
    
    # 3. EXTRACT OPERATORS FROM FACILITY NAMES (before handling missing names)
    print("\n3. Extracting operators from facility names...")
    
    # Define patterns to extract operator names
    operator_patterns = {
        'Google': ['google', 'goog '],
        'Microsoft': ['microsoft', 'msft'],
        'Meta': ['meta', 'facebook', 'fb '],
        'Amazon': ['amazon', 'aws'],
        'Apple': ['apple'],
        'Oracle': ['oracle'],
        'IBM': ['ibm'],
        'Dell': ['dell'],
        'Equinix': ['equinix'],
        'Digital Realty': ['digital realty'],
        'CyrusOne': ['cyrusone', 'cyrus one'],
        'QTS': ['qts'],
        'Switch': ['switch'],
        'Cologix': ['cologix'],
        'CoreSite': ['coresite'],
        'Vantage': ['vantage'],
        'Iron Mountain': ['iron mountain'],
        'Aligned': ['aligned'],
        'EdgeConnex': ['edgeconnex'],
        'DataBank': ['databank'],
        'T5': ['t5 data'],
        'NTT': ['ntt'],
        'Telx': ['telx'],
        'Flexential': ['flexential'],
        'Cyxtera': ['cyxtera'],
        'Datacenter': ['datacenter', 'data center'],  # Generic pattern
    }
    
    def extract_operator_from_name(name):
        """Extract operator from facility name using pattern matching"""
        if pd.isna(name):
            return None
        
        name_lower = name.lower()
        
        # Check each operator pattern
        for operator, patterns in operator_patterns.items():
            for pattern in patterns:
                if pattern in name_lower:
                    # Don't use generic "datacenter" if we already have a better match
                    if operator == 'Datacenter':
                        continue
                    return operator
        
        return None
    
    # Count missing operators before extraction
    missing_before = df_clean['operator'].isnull().sum()
    
    # Extract operators from names where operator is missing
    mask_missing_operator = df_clean['operator'].isnull()
    df_clean.loc[mask_missing_operator, 'operator'] = df_clean.loc[mask_missing_operator, 'name'].apply(extract_operator_from_name)
    
    missing_after = df_clean['operator'].isnull().sum()
    extracted = missing_before - missing_after
    
    print(f"   Missing operators before: {missing_before}")
    print(f"   Extracted {extracted} operators from facility names")
    print(f"   Missing operators after: {missing_after}")
    
    if extracted > 0:
        print(f"\n   Sample extracted operators:")
        newly_filled = mask_missing_operator & df_clean['operator'].notna()
        print(df_clean[newly_filled][['name', 'operator', 'state_abb']].head(10))
    
    # 4. HANDLE MISSING NAMES
    print("\n4. Handling missing names...")
    missing_names_before = df_clean['name'].isnull().sum()
    print(f"   Records with missing names: {missing_names_before}")
    
    # Strategy: Use operator if available, otherwise create descriptive name
    def create_name(row):
        if pd.notna(row['name']):
            return row['name']
        elif pd.notna(row['operator']):
            return f"{row['operator']} Data Center"
        else:
            return f"{row['county']} Data Center"
    
    df_clean['name'] = df_clean.apply(create_name, axis=1)
    
    missing_names_after = df_clean['name'].isnull().sum()
    print(f"   Filled {missing_names_before - missing_names_after} missing names")
    if (missing_names_before - missing_names_after) > 0:
        print(f"   Sample generated names:")
        generated_mask = df_clean['name'].str.contains('Data Center', na=False)
        if generated_mask.any():
            print(df_clean[generated_mask][['name', 'operator', 'county', 'state_abb']].head(3))
    
    # 5. CLEAN OPERATOR FIELD
    print("\n5. Standardizing operator names...")
    
    # Standardize common operator name variations
    operator_standardization = {
        'google': 'Google',
        'Google': 'Google',
        'microsoft': 'Microsoft',
        'Microsoft': 'Microsoft',
        'meta': 'Meta',
        'Meta': 'Meta',
        'facebook': 'Meta',
        'Facebook': 'Meta',
        'amazon': 'Amazon',
        'Amazon': 'Amazon',
        'aws': 'Amazon',
        'AWS': 'Amazon',
        'apple': 'Apple',
        'Apple': 'Apple',
        'oracle': 'Oracle',
        'Oracle': 'Oracle',
    }
    
    # Apply standardization
    df_clean['operator'] = df_clean['operator'].replace(operator_standardization, regex=False)
    
    print(f"   Top operators after standardization:")
    print(df_clean['operator'].value_counts().head(15))
    
    # 6. STANDARDIZE TYPE FIELD
    print("\n6. Standardizing type field...")
    print(f"   Type values before:")
    print(f"   {df_clean['type'].value_counts()}")
    
    # Standardize capitalization
    df_clean['type'] = df_clean['type'].str.capitalize()
    
    print(f"   Type values after:")
    print(f"   {df_clean['type'].value_counts()}")
    
    # 7. VALIDATE AND CLEAN SIZE DATA
    print("\n7. Validating facility size (sqft)...")
    print(f"   Missing sqft: {df_clean['sqft'].isnull().sum()}")
    print(f"   Sqft statistics:")
    print(df_clean['sqft'].describe())
    
    # Flag potentially unrealistic sizes
    very_small = df_clean['sqft'] < 1000
    very_large = df_clean['sqft'] > 10000000
    
    if very_small.any():
        print(f"\n   WARNING: {very_small.sum()} facilities < 1,000 sqft (possibly data entry errors)")
        print(df_clean[very_small][['id', 'name', 'operator', 'sqft', 'type']].head())
    
    if very_large.any():
        print(f"\n   WARNING: {very_large.sum()} facilities > 10M sqft (check if valid)")
        print(df_clean[very_large][['id', 'name', 'operator', 'sqft', 'type']].head())
    
    # Create size categories
    def categorize_size(sqft):
        if pd.isna(sqft):
            return 'Unknown'
        elif sqft < 10000:
            return 'Very Small (<10k sqft)'
        elif sqft < 50000:
            return 'Small (10k-50k sqft)'
        elif sqft < 100000:
            return 'Medium (50k-100k sqft)'
        elif sqft < 500000:
            return 'Large (100k-500k sqft)'
        elif sqft < 1000000:
            return 'Very Large (500k-1M sqft)'
        else:
            return 'Mega (>1M sqft)'
    
    df_clean['size_category'] = df_clean['sqft'].apply(categorize_size)
    print(f"\n   Size categories:")
    print(df_clean['size_category'].value_counts())
    
    # 8. VALIDATE COORDINATES
    print("\n8. Validating coordinates...")
    
    # Check for missing coordinates (should be none based on initial assessment)
    missing_coords = df_clean['lat'].isnull() | df_clean['lon'].isnull()
    if missing_coords.any():
        print(f"   ERROR: {missing_coords.sum()} records with missing coordinates")
        print(df_clean[missing_coords][['id', 'name', 'state', 'county']])
    
    # Validate coordinate ranges for US territories (expanded to include all territories)
    state_bounds = {
        'HI': {'lat_min': 18, 'lat_max': 23, 'lon_min': -161, 'lon_max': -154},
        'AK': {'lat_min': 51, 'lat_max': 72, 'lon_min': -180, 'lon_max': -130},
        'PR': {'lat_min': 17.5, 'lat_max': 18.5, 'lon_min': -67.5, 'lon_max': -65.0},  # Puerto Rico
        'VI': {'lat_min': 17.5, 'lat_max': 18.5, 'lon_min': -65.0, 'lon_max': -64.5},  # US Virgin Islands
        'GU': {'lat_min': 13.0, 'lat_max': 14.0, 'lon_min': 144.5, 'lon_max': 145.5},  # Guam
        'AS': {'lat_min': -14.5, 'lat_max': -14.0, 'lon_min': -171.0, 'lon_max': -169.0},  # American Samoa
        'MP': {'lat_min': 14.0, 'lat_max': 21.0, 'lon_min': 144.0, 'lon_max': 146.0},  # Northern Mariana Islands
        'default': {'lat_min': 24, 'lat_max': 50, 'lon_min': -125, 'lon_max': -65}  # Contiguous US
    }
    
    def check_coord_validity(row):
        """Check if coordinates are valid for the state/territory"""
        state = row['state_abb']
        lat = row['lat']
        lon = row['lon']
        
        # Get appropriate bounds
        bounds = state_bounds.get(state, state_bounds['default'])
        
        in_range = (bounds['lat_min'] <= lat <= bounds['lat_max'] and 
                   bounds['lon_min'] <= lon <= bounds['lon_max'])
        
        return in_range
    
    invalid_coords = ~df_clean.apply(check_coord_validity, axis=1)
    
    if invalid_coords.any():
        print(f"   WARNING: {invalid_coords.sum()} records with coordinates outside expected state bounds")
        print(df_clean[invalid_coords][['id', 'name', 'state_abb', 'county', 'lat', 'lon']])
        print("   Note: These may be valid but in unexpected locations - review manually")
    else:
        print("   All coordinates are within expected bounds for their states/territories")
    
    # 9. CHECK FOR DUPLICATE COORDINATES (multiple facilities at same location)
    print("\n9. Checking for duplicate coordinates...")
    coord_groups = df_clean.groupby(['lat', 'lon']).size()
    duplicates = coord_groups[coord_groups > 1]
    
    if len(duplicates) > 0:
        print(f"   Found {len(duplicates)} coordinate pairs with multiple facilities")
        print(f"   Total facilities sharing coordinates: {duplicates.sum()}")
        print("\n   Sample locations with multiple facilities:")
        
        for (lat, lon), count in duplicates.head(3).items():
            print(f"\n   Location ({lat:.4f}, {lon:.4f}): {count} facilities")
            facilities = df_clean[(df_clean['lat'] == lat) & (df_clean['lon'] == lon)]
            print(facilities[['id', 'name', 'operator', 'type', 'sqft']])
    else:
        print("   No duplicate coordinates found")
    
    # 10. VALIDATE STATE CONSISTENCY
    print("\n10. Validating state data...")
    
    # Check if state and state_abb are consistent
    state_mapping = df_clean.groupby('state')['state_abb'].unique()
    inconsistent = [s for s in state_mapping.index if len(state_mapping[s]) > 1]
    
    if inconsistent:
        print(f"   WARNING: Inconsistent state/state_abb mappings for: {inconsistent}")
        for state in inconsistent:
            print(f"      {state}: {state_mapping[state]}")
    else:
        print("   State/state_abb mappings are consistent")
    
    print(f"\n   Facilities by state/territory (top 10):")
    print(df_clean['state'].value_counts().head(10))
    
    # 11. CREATE DATA QUALITY FLAGS
    print("\n11. Creating data quality flags...")
    df_clean['has_operator'] = df_clean['operator'].notna()
    df_clean['operator_extracted'] = mask_missing_operator & df_clean['operator'].notna()  # Track extracted operators
    df_clean['has_size'] = df_clean['sqft'].notna()
    df_clean['has_ref'] = df_clean['ref'].notna()
    df_clean['is_campus'] = df_clean['type'] == 'Campus'
    df_clean['is_building'] = df_clean['type'] == 'Building'
    df_clean['name_is_generated'] = df_clean['name'].str.contains('Data Center', na=False)
    
    # Overall data completeness score (0-3)
    df_clean['data_completeness'] = (
        df_clean['has_operator'].astype(int) +
        df_clean['has_size'].astype(int) +
        df_clean['has_ref'].astype(int)
    )
    
    print(f"\n   Data quality summary:")
    print(f"   Records with operator: {df_clean['has_operator'].sum()} ({df_clean['has_operator'].sum()/len(df_clean)*100:.1f}%)")
    print(f"   Operators extracted from name: {df_clean['operator_extracted'].sum()}")
    print(f"   Records with size: {df_clean['has_size'].sum()} ({df_clean['has_size'].sum()/len(df_clean)*100:.1f}%)")
    print(f"   Records with reference: {df_clean['has_ref'].sum()} ({df_clean['has_ref'].sum()/len(df_clean)*100:.1f}%)")
    print(f"   Campus type: {df_clean['is_campus'].sum()}")
    print(f"   Building type: {df_clean['is_building'].sum()}")
    print(f"\n   Data completeness distribution:")
    print(df_clean['data_completeness'].value_counts().sort_index())
    
    print(f"\nFinal shape: {df_clean.shape}")
    
    return df_clean

# Run the cleaning
dc_osdc_clean = clean_osdc(dc_osdc)

# 1. STANDARDIZE COLUMN NAMES
dc_osdc_clean = dc_osdc_clean.rename(columns={
    'lon': 'longitude',
    'lat': 'latitude',
    'sqft': 'facility_size_sqft'
})


# Save cleaned version
dc_osdc_clean.to_csv('/Users/zdiener/data-center-classification/data/processed/osdc_cleaned.csv', index=False)
print("\nCleaned OSDC data saved to: data/processed/osdc_cleaned.csv")

# Show summary comparison
print("\n" + "="*60)
print("OSDC CLEANING SUMMARY")
print("="*60)
print(f"\nOriginal records: {len(dc_osdc)}")
print(f"Cleaned records: {len(dc_osdc_clean)}")
print(f"\nMissing data handled:")
print(f"  Names: {dc_osdc['name'].isnull().sum()} -> {dc_osdc_clean['name'].isnull().sum()}")
print(f"  Operators: {dc_osdc['operator'].isnull().sum()} (kept as-is)")
print(f"  Sqft: {dc_osdc['sqft'].isnull().sum()} (kept as-is)")
print(f"\nNew fields created:")
print(f"  - size_category")
print(f"  - has_operator, has_size, has_ref")
print(f"  - is_campus, is_building")
print(f"  - name_is_generated")
print(f"  - data_completeness")

Starting OSDC cleaning...
Initial shape: (1242, 13)

1. Checking for duplicates...
   Removed 0 duplicate rows

2. Stripping whitespace from text fields...
   Whitespace stripped from 7 fields

3. Extracting operators from facility names...
   Missing operators before: 458
   Extracted 62 operators from facility names
   Missing operators after: 396

   Sample extracted operators:
                              name       operator state_abb
4       Apple - Maiden Data Center          Apple        NC
12                    CoreSite BO1       CoreSite        MA
52                    IBM Building            IBM        NY
57      CoreSite Reston Campus VA1       CoreSite        VA
61     CyrusOne Cincinnati-Lebanon       CyrusOne        OH
70   Iron Mountain Records Storage  Iron Mountain        AZ
80      Aligned Energy Data Center        Aligned        AZ
85        DataBank Pittsburgh PIT2       DataBank        PA
104    CyrusOne Chicago Aurora CME       CyrusOne        IL
108       DataBa